# Structure feasibility: exact identifier-repair amendment

This CPU-only notebook preserves the frozen Phase-1 audit and retries only its failed AlphaFold accessions. A replacement accession is accepted exclusively through an exact complete-sequence match from the official AlphaFold DB or UniProt APIs. The original 95% gates are unchanged, no model is fitted, and no independent or external test is accessed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
import json
print('CPU-only identifier repair; a GPU is not required.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

In [ ]:
CONFIG = Path('experiment/configs/structure_context_feasibility_identifier_repair_v2.json')
SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')
PHASE1_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/structure/structure-context-feasibility-v1')
PHASE1_CACHE = PHASE1_DIR / 'alphafold_training_site_metadata.json'
PHASE1_REPORT = PHASE1_DIR / 'structure_feasibility.json'
OUTPUT_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/structure/structure-context-feasibility-identifier-repair-v2')
METADATA_CACHE = OUTPUT_DIR / 'alphafold_training_site_metadata_repaired.json'
REPORT_PATH = OUTPUT_DIR / 'structure_feasibility.json'

for required in (SEQUENCE_CACHE, PHASE1_CACHE, PHASE1_REPORT):
    if not required.exists():
        raise FileNotFoundError(f'Missing required Phase-1 artifact: {required}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Frozen Phase-1 report:', PHASE1_REPORT)
print('Separate repair cache:', METADATA_CACHE)
print('Separate amended report:', REPORT_PATH)

## Retry failed accessions with exact-sequence identifier repair

The first execution seeds a new cache from the frozen Phase-1 cache. After a runtime interruption or retryable network error, re-execution retains completed accessions and continues from the Drive-backed cache.

In [ ]:
run_live([
    sys.executable, '-u',
    'experiment/scripts/fetch_alphafold_site_confidence.py',
    '--data-dir', 'replication/MMUbiPred',
    '--sequence-cache', SEQUENCE_CACHE,
    '--seed-cache', PHASE1_CACHE,
    '--output', METADATA_CACHE,
    '--max-workers', '6',
    '--checkpoint-every', '25',
])

## Freeze the amended training-only audit

This audit stage is enabled only after retrieval reports `retrieval_complete`. It applies the same three 95% coverage gates as Phase 1.

In [ ]:
if not REPORT_PATH.exists():
    run_live([
        sys.executable, '-u',
        'experiment/scripts/audit_structure_feasibility.py',
        '--config', CONFIG,
        '--metadata-cache', METADATA_CACHE,
        '--output-dir', OUTPUT_DIR,
    ])
else:
    print('Existing frozen amended report found; it was not overwritten.')

report = json.loads(REPORT_PATH.read_text())
display({
    'status': report['status'],
    'training_authorized': report['training_authorized'],
    'coverage_checks': report['coverage_checks'],
    'coverage': report['coverage'],
    'identifier_resolution': report['identifier_resolution'],
    'reason_counts': report['reason_counts'],
    'site_plddt': report['site_plddt'],
    'released_independent_or_external_test_accessed': report['released_independent_or_external_test_accessed'],
})

## Feasibility decision record

The displayed dictionary forms the feasibility decision record. Coordinate retrieval and model training are authorized only when `training_authorized` is `True`; any failed gate ends the structural-residual proposal under the frozen protocol.